# Encoder Block

                 Input
                   │
                   ▼
         Multi Head Attention
                   │
                   ▼
          Add (Residual)
                   │
                   ▼
             LayerNorm
                   │
                   ▼
          Feed Forward
                   │
                   ▼
          Add (Residual)
                   │
                   ▼
             LayerNorm
                   │
                   ▼
               Output

##MultiHead Attention

In [1]:
import math
import torch
import torch.nn as nn

In [15]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model=d_model
    self.num_heads=num_heads
    self.head_dim=d_model//num_heads

    self.query=nn.Linear(d_model,d_model,bias=False)
    self.key=nn.Linear(d_model,d_model,bias=False)
    self.value=nn.Linear(d_model,d_model,bias=False)
    self.fc=nn.Linear(d_model,d_model,bias=False)

  def forward(self,x):
    B,S,E=x.shape #(batch,seq_len,embed_dim)

    Q=self.query(x) #(batch,seq_len,embed_dim)
    K=self.key(x) #(batch,seq_len,embed_dim)
    V=self.value(x) #(batch,seq_len,embed_dim)

    #split into heads
    Q=Q.view(B,S,self.num_heads,self.head_dim) #(batch,seq_len,num_heads,head_dim)
    K=K.view(B,S,self.num_heads,self.head_dim) #(batch,seq_len,num_heads,head_dim)
    V=V.view(B,S,self.num_heads,self.head_dim)

    Q=Q.transpose(1,2)
    K=K.transpose(1,2)
    V=V.transpose(1,2)
    #shape of Q,K,V (batch,num_heads,seq_len,head_dim)

    #scores
    scores=torch.matmul(
        Q,K.transpose(-2,-1)
    )

    scores=scores/math.sqrt(self.head_dim)

    attention=torch.softmax(
        scores,dim=-1
    )

    #context
    context=torch.matmul(
        attention,V
    )

    context=context.transpose(1,2)
    #before transpose(B,H,S,D)
    #after transpose(B,S,H,D)

    context=context.contiguous()

    context=context.view(B,S,self.d_model)
    #shape of context (batch,seq_len,embed_dim)

    output=self.fc(context)

    return output



## LayerNormalization

In [16]:
class LayerNormalization(nn.Module):
  def __init__(self,d_model,eps=1e-5):
    super().__init__()

    self.gamma=nn.Parameter(torch.ones(d_model))
    self.beta=nn.Parameter(torch.zeros(d_model))
    self.eps=eps

  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    var=x.var(dim=-1,keepdim=True)

    x=(x-mean)/torch.sqrt(var+self.eps)

    x=self.gamma*x+self.beta

    return x


## Feed Forward Network

In [17]:
class FeedForward(nn.Module):
  def __init__(self,d_model,hidden_dim):
    super().__init__()
    self.fc1=nn.Linear(d_model,hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim,d_model)

  def forward(self,x):
    out=self.fc1(x)
    out=self.relu(out)
    out=self.fc2(out)

    return out

##Encoder Block

In [18]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,d_model,num_heads,hidden_dim):
    super().__init__()

    self.attention=MultiHeadAttention(d_model,num_heads)
    self.norm1=LayerNormalization(d_model)
    self.ffn=FeedForward(d_model,hidden_dim)
    self.norm2=LayerNormalization(d_model)

  def forward(self,x):
    attention_output=self.attention(x) #attention
    x=x+attention_output #residual
    x=self.norm1(x) #layernormalization

    ffn_output=self.ffn(x) #FeedForwardNetwork
    x=x+ffn_output #residual
    x=self.norm2(x) #layernormalization

    return x



In [19]:
encoder=TransformerEncoderBlock(
    d_model=512,
    num_heads=8,
    hidden_dim=2048
)
x=torch.randn(
    2,10,512
)
output=encoder(x)
print(output.shape)

torch.Size([2, 10, 512])


# Let a Look Shape In Encoder



```
                  Input
             (2,4,8)
                  │
                  ▼
        Multi Head Attention
             (2,4,8)
                  │
                  ▼
           Residual Add
                  │
                  ▼
            LayerNorm
             (2,4,8)
                  │
                  ▼
         Feed Forward Network
         8 → 32 → 8
                  │
                  ▼
           Residual Add
                  │
                  ▼
            LayerNorm
             (2,4,8)
                  │
                  ▼
               Output
```





---



---
Z.I. Turjo
